In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')   

In [2]:
import pickle

with open("general_conditions", "rb") as fp:
    general_conditions = pickle.load(fp)
    
with open("non_general_conditions", "rb") as fp:
    non_general_conditions = pickle.load(fp)

In [3]:
print(len(general_conditions))
print(len(non_general_conditions))

100
6577


In [5]:
#from datasets import load_dataset, Dataset, load_from_disk
import pandas as pd
path = 'test_result/generate/'

df_llama3_freq = pd.read_csv(path + 'llama3_sample_frequency.csv')
df_llama3_length = pd.read_csv(path + 'llama3_sample_length.csv')
df_llama3_random = pd.read_csv(path + 'llama3_sample_random.csv')

df_llama3_instruct_freq = pd.read_csv(path + 'llama3-instruct_sample_frequency.csv')
df_llama3_instruct_length = pd.read_csv(path + 'llama3-instruct_sample_length.csv')
df_llama3_instruct_random = pd.read_csv(path + 'llama3-instruct_sample_random.csv')

df_llama3_1b_freq = pd.read_csv(path + 'llama3-1b_sample_frequency.csv')
df_llama3_1b_length = pd.read_csv(path + 'llama3-1b_sample_length.csv')
df_llama3_1b_random = pd.read_csv(path + 'llama3-1b_sample_random.csv')

In [4]:
#from datasets import load_dataset, Dataset, load_from_disk
import pandas as pd

df_biomistral_freq = pd.read_csv(path + 'biomistral_sample_frequency_generate_fake_False.csv')
df_biomistral_length = pd.read_csv(path + 'biomistral_sample_length_generate_fake_False.csv')
df_biomistral_random = pd.read_csv(path + 'biomistral_sample_random_generate_fake_False.csv')

df_meditron_freq = pd.read_csv(path + 'meditron_sample_frequency_generate_fake_False.csv')
df_meditron_length = pd.read_csv(path + 'meditron_sample_length_generate_fake_False.csv')
df_meditron_random = pd.read_csv(path + 'meditron_sample_random_generate_fake_False.csv')

df_medalpaca_freq = pd.read_csv(path + 'medalpaca_sample_frequency_generate_fake_False.csv')
df_medalpaca_length = pd.read_csv(path + 'medalpaca_sample_length_generate_fake_False.csv')
df_medalpaca_random = pd.read_csv(path + 'medalpaca_sample_random_generate_fake_False.csv')

In [17]:
def general_process_em(df):
    df.columns = ['SUBJECT_ID','output','condition','gender','name']
    df_tmp = df[['output', 'condition']]
    
    df_tmp['em_n'] = 0
    df_tmp['recall_em'] = 0
    is_in = {}
    for i in (range(len(df_tmp))):
        condition_list = df_tmp['condition'][i].replace("[", "").replace("]", "").replace("'", "").split(", ")
        target_list = list(set(condition_list) - set(non_general_conditions))
        
        opt_em_n = 0
        for j in range(len(target_list)):
            target = target_list[j].lower()
            
            if target in df_tmp.output[i].lower():
                opt_em_n += 1
                
                is_in[target] = is_in.get(target, 0) + 1

        
        df_tmp.loc[i, 'em_n'] = int(opt_em_n)
        df_tmp.loc[i, 'recall_em'] = float(opt_em_n) / len(target_list)

        is_in_df = pd.DataFrame(list(is_in.items()), columns=['condition', 'count'])
        is_in_df = is_in_df.sort_values(by='count', ascending=False)

    return np.average(df_tmp.recall_em)#, is_in_df

In [18]:
def non_general_process_em(df):
    df.columns = ['SUBJECT_ID','output','condition','gender','name']
    df_tmp = df[['output', 'condition']]
    
    df_tmp['em_n'] = 0
    df_tmp['recall_em'] = 0
    is_in = {}
    for i in (range(len(df_tmp))):
        condition_list = df_tmp['condition'][i].replace("[", "").replace("]", "").replace("'", "").split(", ")
        target_list = list(set(condition_list) - set(general_conditions))
        
        opt_em_n = 0
        for j in range(len(target_list)):
            target = target_list[j].lower()
            
            if target in df_tmp.output[i].lower():
                opt_em_n += 1
                
                is_in[target] = is_in.get(target, 0) + 1

        df_tmp.loc[i, 'em_n'] = int(opt_em_n)
        df_tmp.loc[i, 'recall_em'] = float(opt_em_n) / len(target_list)

        is_in_df = pd.DataFrame(list(is_in.items()), columns=['condition', 'count'])
        is_in_df = is_in_df.sort_values(by='count', ascending=False)

    return np.average(df_tmp.recall_em)#, is_in_df

In [19]:
print("### General Exact Match ###")
print("[Biomistral]")
print("biomistral_freq: ", general_process_em(df_biomistral_freq))
print("biomistral_leng: ", general_process_em(df_biomistral_length))
print("biomistral_rand: ", general_process_em(df_biomistral_random))
print("=========================")
print("[Meditron]")
print("meditron_freq: ", general_process_em(df_meditron_freq))
print("meditron_leng: ", general_process_em(df_meditron_length))
print("meditron_rand: ", general_process_em(df_meditron_random))
print("=========================")
print("[Medalpaca]")
print("medalpaca_freq: ", general_process_em(df_medalpaca_freq))
print("medalpaca_leng: ", general_process_em(df_medalpaca_length))
print("medalpaca_rand: ", general_process_em(df_medalpaca_random))

### General Exact Match ###
[Biomistral]
biomistral_freq:  0.016919645828072427
biomistral_leng:  0.01783533097750933
biomistral_rand:  0.01679746502108233
[Meditron]
meditron_freq:  0.003347644310820099
meditron_leng:  0.0032379287487126138
meditron_rand:  0.003211302129258983
[Medalpaca]
medalpaca_freq:  0.0003739520641003692
medalpaca_leng:  0.0003882986817322791
medalpaca_rand:  0.0004113648349838767


In [20]:
print("### Non-General Exact Match ###")
print("[Biomistral]")
print("biomistral_freq: ", non_general_process_em(df_biomistral_freq))
print("biomistral_leng: ", non_general_process_em(df_biomistral_length))
print("biomistral_rand: ", non_general_process_em(df_biomistral_random))
print("=========================")
print("[Meditron]")
print("meditron_freq: ", non_general_process_em(df_meditron_freq))
print("meditron_leng: ", non_general_process_em(df_meditron_length))
print("meditron_rand: ", non_general_process_em(df_meditron_random))
print("=========================")
print("[Medalpaca]")
print("medalpaca_freq: ", non_general_process_em(df_medalpaca_freq))
print("medalpaca_leng: ", non_general_process_em(df_medalpaca_length))
print("medalpaca_rand: ", non_general_process_em(df_medalpaca_random))

### Non-General Exact Match ###
[Biomistral]
biomistral_freq:  0.016713883760499378
biomistral_leng:  0.01739798872451528
biomistral_rand:  0.016429685443527263
[Meditron]
meditron_freq:  0.003337388751997376
meditron_leng:  0.0029819652117741456
meditron_rand:  0.003035830465121215
[Medalpaca]
medalpaca_freq:  0.0003090788583048645
medalpaca_leng:  0.00034499294927381105
medalpaca_rand:  0.000424219707474585


In [20]:
print("### General Exact Match ###")
print("[LLaMA3 8B]")
print("llama3_freq: ", general_process_em(df_llama3_freq))
print("llama3_leng: ", general_process_em(df_llama3_length))
print("llama3_rand: ", general_process_em(df_llama3_random))
print("=========================")
print("[LLaMA3 Instruct 8B]")
print("llama3_instruct_freq: ", general_process_em(df_llama3_instruct_freq))
print("llama3_instruct_leng: ", general_process_em(df_llama3_instruct_length))
print("llama3_instruct_rand: ", general_process_em(df_llama3_instruct_random))
print("=========================")
print("[LLaMA3 1B]")
print("llama3_1b_freq: ", general_process_em(df_llama3_1b_freq))
print("llama3_1b_leng: ", general_process_em(df_llama3_1b_length))
print("llama3_1b_rand: ", general_process_em(df_llama3_1b_random))

### General Exact Match ###
[LLaMA3 7B]
llama3_freq:  0.009603571946317828
llama3_leng:  0.008459204710588487
llama3_rand:  0.009158063346358072
[LLaMA3 Instruct 7B]
llama3_instruct_freq:  0.009053177465496798
llama3_instruct_leng:  0.009692053421580192
llama3_instruct_rand:  0.010129231711636072
[LLaMA3 1B]
llama3_1b_freq:  0.0012760082891364677
llama3_1b_leng:  0.00157076424132347
llama3_1b_rand:  0.0014565253635662916


In [22]:
print("### Non-General Exact Match ###")
print("[LLaMA3 8B]")
print("llama3_freq: ", non_general_process_em(df_llama3_freq))
print("llama3_leng: ", non_general_process_em(df_llama3_length))
print("llama3_rand: ", non_general_process_em(df_llama3_random))
print("=========================")
print("[LLaMA3 Instruct 8B]")
print("llama3_instruct_freq: ", non_general_process_em(df_llama3_instruct_freq))
print("llama3_instruct_leng: ", non_general_process_em(df_llama3_instruct_length))
print("llama3_instruct_rand: ", non_general_process_em(df_llama3_instruct_random))
print("=========================")
print("[LLaMA3 1B]")
print("llama3_1b_freq: ", non_general_process_em(df_llama3_1b_freq))
print("llama3_1b_leng: ", non_general_process_em(df_llama3_1b_length))
print("llama3_1b_rand: ", non_general_process_em(df_llama3_1b_random))

### Non-General Exact Match ###
[LLaMA3 8B]
llama3_freq:  0.009518882100360698
llama3_leng:  0.008168455872342679
llama3_rand:  0.009153161855825646
[LLaMA3 Instruct 8B]
llama3_instruct_freq:  0.009119508000291217
llama3_instruct_leng:  0.009712870519172462
llama3_instruct_rand:  0.01012696367672522
[LLaMA3 1B]
llama3_1b_freq:  0.001206296113625614
llama3_1b_leng:  0.0013450260008299403
llama3_1b_rand:  0.0012249194207962022


In [23]:
def general_process_partial(df, n):
    df.columns = ['SUBJECT_ID','output','condition','gender','name']
    df_tmp = df[['output', 'condition']]
    
    df_tmp['em_n'] = 0
    df_tmp['recall_em'] = 0
    is_in = {}
    for i in (range(len(df_tmp))):
        condition_list = df_tmp['condition'][i].replace("[", "").replace("]", "").replace("'", "").split(", ")
        target_list = list(set(condition_list) - set(non_general_conditions))
        split_by_space = [cond.split() for cond in target_list]
        flattened_list = [word for condition in split_by_space for word in condition]
        flattened_list = list(set(flattened_list))
        flattened_list = [word for word in flattened_list if len(word) >= n]
        
        opt_em_n = 0
        
        for j in range(len(flattened_list)):
            target = flattened_list[j].lower()
            
            if target in df_tmp.output[i].lower():
                opt_em_n += 1
                is_in[target] = is_in.get(target, 0) + 1
        
        df_tmp.loc[i, 'em_n'] = int(opt_em_n)
        df_tmp.loc[i, 'recall_em'] = float(opt_em_n) / len(flattened_list)

        is_in_df = pd.DataFrame(list(is_in.items()), columns=['condition', 'count'])
        is_in_df = is_in_df.sort_values(by='count', ascending=False)

    return np.average(df_tmp.recall_em)

In [21]:
def non_general_process_partial(df, n):
    df.columns = ['SUBJECT_ID','output','condition','gender','name']
    df_tmp = df[['output', 'condition']]
    
    df_tmp['em_n'] = 0
    df_tmp['recall_em'] = 0
    is_in = {}
    for i in (range(len(df_tmp))):
        condition_list = df_tmp['condition'][i].replace("[", "").replace("]", "").replace("'", "").split(", ")
        target_list = list(set(condition_list) - set(general_conditions))
        split_by_space = [cond.split() for cond in target_list]
        flattened_list = [word for condition in split_by_space for word in condition]
        flattened_list = list(set(flattened_list))
        flattened_list = [word for word in flattened_list if len(word) >= n]
        
        opt_em_n = 0
        
        for j in range(len(flattened_list)):
            target = flattened_list[j].lower()
            
            if target in df_tmp.output[i].lower():
                opt_em_n += 1
                is_in[target] = is_in.get(target, 0) + 1
        
        df_tmp.loc[i, 'em_n'] = int(opt_em_n)
        df_tmp.loc[i, 'recall_em'] = float(opt_em_n) / len(flattened_list)

        is_in_df = pd.DataFrame(list(is_in.items()), columns=['condition', 'count'])
        is_in_df = is_in_df.sort_values(by='count', ascending=False)

    return np.average(df_tmp.recall_em)

In [24]:
n = 4
print("### set n=4 ###")
print("### General Partial word Match ###")
print("[Biomistral]")
print("biomistral_freq: ", general_process_partial(df_biomistral_freq, n))
print("biomistral_leng: ", general_process_partial(df_biomistral_length, n))
print("biomistral_rand: ", general_process_partial(df_biomistral_random, n))
print("=========================")
print("[Meditron]")
print("meditron_freq: ", general_process_partial(df_meditron_freq, n))
print("meditron_leng: ", general_process_partial(df_meditron_length, n))
print("meditron_rand: ", general_process_partial(df_meditron_random, n))
print("=========================")
print("[Medalpaca]")
print("medalpaca_freq: ", general_process_partial(df_medalpaca_freq, n))
print("medalpaca_leng: ", general_process_partial(df_medalpaca_length, n))
print("medalpaca_rand: ", general_process_partial(df_medalpaca_random, n))

### set n=4 ###
### General Partial word Match ###
[Biomistral]
biomistral_freq:  0.12539969912455953
biomistral_leng:  0.1272050371276636
biomistral_rand:  0.12423573593117367
[Meditron]
meditron_freq:  0.06523437457405842
meditron_leng:  0.06639692270159026
meditron_rand:  0.06701564944024269
[Medalpaca]
medalpaca_freq:  0.018059976845757372
medalpaca_leng:  0.0191469461947714
medalpaca_rand:  0.019093107909538404


In [27]:
n = 4
print("### set n=4 ###")
print("### General Partial word Match ###")
print("[LLaMA3 8B]")
print("llama3_freq: ", general_process_partial(df_llama3_freq, n))
print("llama3_leng: ", general_process_partial(df_llama3_length, n))
print("llama3_rand: ", general_process_partial(df_llama3_random, n))
print("=========================")
print("[LLaMA3 Instruct 8B]")
print("llama3_instruct_freq: ", general_process_partial(df_llama3_instruct_freq, n))
print("llama3_instruct_leng: ", general_process_partial(df_llama3_instruct_length, n))
print("llama3_instruct_rand: ", general_process_partial(df_llama3_instruct_random, n))
print("=========================")
print("[LLaMA3 1B]")
print("llama3_1b_freq: ", general_process_partial(df_llama3_1b_freq, n))
print("llama3_1b_leng: ", general_process_partial(df_llama3_1b_length, n))
print("llama3_1b_rand: ", general_process_partial(df_llama3_1b_random, n))

### set n=4 ###
### General Partial word Match ###
[LLaMA3 8B]
llama3_freq:  0.0991878042792867
llama3_leng:  0.10359703258459621
llama3_rand:  0.09893012462812112
[LLaMA3 Instruct 8B]
llama3_instruct_freq:  0.09859920887154594
llama3_instruct_leng:  0.1023238322148253
llama3_instruct_rand:  0.10308405317824436
[LLaMA3 1B]
llama3_1b_freq:  0.034277806515054375
llama3_1b_leng:  0.03411181379919186
llama3_1b_rand:  0.03447342610073451


In [25]:
n = 4
print("### set n=4 ###")
print("### Non-General Partial word Match ###")
print("[Biomistral]")
print("biomistral_freq: ", non_general_process_partial(df_biomistral_freq, n))
print("biomistral_leng: ", non_general_process_partial(df_biomistral_length, n))
print("biomistral_rand: ", non_general_process_partial(df_biomistral_random, n))
print("=========================")
print("[Meditron]")
print("meditron_freq: ", non_general_process_partial(df_meditron_freq, n))
print("meditron_leng: ", non_general_process_partial(df_meditron_length, n))
print("meditron_rand: ", non_general_process_partial(df_meditron_random, n))
print("=========================")
print("[Medalpaca]")
print("medalpaca_freq: ", non_general_process_partial(df_medalpaca_freq, n))
print("medalpaca_leng: ", non_general_process_partial(df_medalpaca_length, n))
print("medalpaca_rand: ", non_general_process_partial(df_medalpaca_random, n))

### set n=4 ###
### Non-General Partial word Match ###
[Biomistral]
biomistral_freq:  0.1259570666089609
biomistral_leng:  0.12767093645589406
biomistral_rand:  0.12477066943328755
[Meditron]
meditron_freq:  0.0672561183219865
meditron_leng:  0.06814176818235956
meditron_rand:  0.0688942566903599
[Medalpaca]
medalpaca_freq:  0.018401602649936153
medalpaca_leng:  0.019485322844986244
medalpaca_rand:  0.01951735329932089


In [28]:
n = 4
print("### set n=4 ###")
print("### Non-General Partial word Match ###")
print("[LLaMA3 8B]")
print("llama3_freq: ", non_general_process_partial(df_llama3_freq, n))
print("llama3_leng: ", non_general_process_partial(df_llama3_length, n))
print("llama3_rand: ", non_general_process_partial(df_llama3_random, n))
print("=========================")
print("[LLaMA3 Instruct 8B]")
print("llama3_instruct_freq: ", non_general_process_partial(df_llama3_instruct_freq, n))
print("llama3_instruct_leng: ", non_general_process_partial(df_llama3_instruct_length, n))
print("llama3_instruct_rand: ", non_general_process_partial(df_llama3_instruct_random, n))
print("=========================")
print("[LLaMA3 1B]")
print("llama3_1b_freq: ", non_general_process_partial(df_llama3_1b_freq, n))
print("llama3_1b_leng: ", non_general_process_partial(df_llama3_1b_length, n))
print("llama3_1b_rand: ", non_general_process_partial(df_llama3_1b_random, n))

### set n=4 ###
### Non-General Partial word Match ###
[LLaMA3 8B]
llama3_freq:  0.10118044859194024
llama3_leng:  0.10566222345551968
llama3_rand:  0.10107602893695247
[LLaMA3 Instruct 8B]
llama3_instruct_freq:  0.10083714859915895
llama3_instruct_leng:  0.1044050968231961
llama3_instruct_rand:  0.10513116143966354
[LLaMA3 1B]
llama3_1b_freq:  0.03475160091887019
llama3_1b_leng:  0.03467909488899393
llama3_1b_rand:  0.03489544297987926


In [6]:
def process_em(df):

    df['em_n'] = 0
    df['recall_em'] = 0
    is_in = {}
    for i in (range(len(df))):
        condition_list = df['condition'][i].replace("[", "").replace("]", "").replace("'", "").split(", ")
        target_list = []
        for condition in set(condition_list):
            if 'NEC/NOS' in condition:
                condition = condition.replace('NEC/NOS', '').strip()
            elif 'NOS' in condition:
                condition = condition.replace('NOS', '').strip()
            elif 'NEC' in condition:
                condition = condition.replace('NEC', '').strip()
            if condition: 
                target_list.append(condition)
        
        opt_em_n = 0
        for j in range(len(target_list)):
            target = target_list[j].lower()
            
            if target in df.output[i].lower():
                opt_em_n += 1
                
                is_in[target] = is_in.get(target, 0) + 1

        
        df.loc[i, 'em_n'] = int(opt_em_n)
        df.loc[i, 'recall_em'] = float(opt_em_n) / len(target_list)

        is_in_df = pd.DataFrame(list(is_in.items()), columns=['condition', 'count'])
        is_in_df = is_in_df.sort_values(by='count', ascending=False)

    return df, is_in_df

In [8]:
def count_general_ratio(df):
    a, b = process_em(df)
    b['is_general'] = b['condition'].isin(general_conditions)
    return len(b[b.is_general==True]) / len(b)


print("### Count General Ratio - Exact Match ###")
print("[LLaMA3 8B]")
print("llama3_freq: ", count_general_ratio(df_llama3_freq))
print("llama3_leng: ", count_general_ratio(df_llama3_length))
print("llama3_rand: ", count_general_ratio(df_llama3_random))
print("=========================")
print("[LLaMA3 8B Instruction]")
print("llama3_instruct_freq: ", count_general_ratio(df_llama3_instruct_freq))
print("llama3_instruct_leng: ", count_general_ratio(df_llama3_instruct_length))
print("llama3_instruct_rand: ", count_general_ratio(df_llama3_instruct_random))
print("=========================")
print("[LLaMA3 1B]")
print("llama3_1b_freq: ", count_general_ratio(df_llama3_1b_freq))
print("llama3_1b_leng: ", count_general_ratio(df_llama3_1b_length))
print("llama3_1b_rand: ", count_general_ratio(df_llama3_1b_random))

### Count General Ratio - Exact Match ###
[LLaMA3 8B]
llama3_freq:  0.13846153846153847
llama3_leng:  0.2608695652173913
llama3_rand:  0.1044776119402985
[LLaMA3 8B Instruction]
llama3_instruct_freq:  0.13725490196078433
llama3_instruct_leng:  0.11904761904761904
llama3_instruct_rand:  0.18518518518518517
[LLaMA3 1B]
llama3_1b_freq:  0.2
llama3_1b_leng:  0.2222222222222222
llama3_1b_rand:  0.15217391304347827
